In [1]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", None)

REPO_ROOT = Path("..").resolve()
DATA_RAW = REPO_ROOT / "data" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

JOBS_FILE = "jobs_dataset.csv"
RESUMES_FILE = "resumes_dataset.csv"

jobs = pd.read_csv(DATA_RAW / JOBS_FILE)
resumes = pd.read_csv(DATA_RAW / RESUMES_FILE)

print("jobs shape:", jobs.shape)
print("resumes shape:", resumes.shape)

jobs shape: (1068, 7)
resumes shape: (1200, 14)


In [2]:
# -----------------------------
# Basic text cleaning helpers
# -----------------------------

def normalize_text(x):
    """
    Convert values to clean strings.
    - Handles NaN
    - Removes extra spaces
    """
    if pd.isna(x):
        return ""
    x = str(x).replace("\u00a0", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()


# Common weak suffixes that do not add much meaning
SKILL_NOISE_SUFFIXES = [
    " basics", " fundamental", " fundamentals", " beginner",
    " introductory", " intro", " advanced"
]

def clean_skill_token(token: str) -> str:
    """
    Normalize one skill token.
    Example:
    'Python Basics' -> 'python'
    """
    s = normalize_text(token).lower()
    s = re.sub(r"\s+", " ", s).strip()

    for suf in SKILL_NOISE_SUFFIXES:
        if s.endswith(suf):
            s = s[: -len(suf)].strip()

    return s


def parse_skills(x):
    """
    Parse skill strings into clean Python lists.
    Handles delimiters such as:
    - comma
    - semicolon
    - slash
    - pipe
    """
    if pd.isna(x):
        return []

    raw = str(x)
    parts = re.split(r"[;,/|]", raw)

    skills = []
    seen = set()

    for p in parts:
        s = clean_skill_token(p)
        if s and s not in seen:
            seen.add(s)
            skills.append(s)

    return skills


def parse_years_range(x: str):
    """
    Convert job experience text into numeric min/max range.
    Examples:
    '0-1' -> (0, 1)
    '10+' -> (10, None)
    '5 years' -> (5, 5)
    """
    if x is None:
        return (None, None)

    s = str(x).lower().strip()
    s = s.replace("years", "").replace("year", "").strip()
    s = s.replace("–", "-")

    if not s:
        return (None, None)

    # Pattern like 10+
    m = re.match(r"(\d+)\s*\+", s)
    if m:
        return (int(m.group(1)), None)

    # Pattern like 2-4
    m = re.match(r"(\d+)\s*-\s*(\d+)", s)
    if m:
        return (int(m.group(1)), int(m.group(2)))

    # Pattern like 3
    m = re.match(r"(\d+)", s)
    if m:
        v = int(m.group(1))
        return (v, v)

    return (None, None)


def normalize_title(title):
    """
    Normalize titles so role matching is easier.
    Removes level qualifiers such as:
    - fresher
    - experienced
    - junior
    - senior
    - entry level
    """
    if pd.isna(title):
        return ""

    t = str(title).lower()

    # Remove anything after a dash
    t = re.sub(r"-.*", "", t)

    # Remove level indicators
    t = re.sub(
        r"\b(fresher|experienced|senior|junior|mid|lead|entry level|entry-level|associate)\b",
        "",
        t
    )

    t = re.sub(r"\s+", " ", t)
    return t.strip()

In [3]:
# -----------------------------
# Build a cleaner jobs table
# -----------------------------

jobs_exp = jobs.copy()

# Fill any missing titles with empty string
jobs_exp["Title"] = jobs_exp["Title"].fillna("")

# Canonical fields
jobs_exp["job_id"] = jobs_exp["JobID"].astype(str)
jobs_exp["job_title"] = jobs_exp["Title"].map(normalize_text)
jobs_exp["normalized_job_title"] = jobs_exp["job_title"].map(normalize_title)

jobs_exp["experience_level"] = jobs_exp["ExperienceLevel"].map(normalize_text).str.lower()
jobs_exp["years_of_experience"] = jobs_exp["YearsOfExperience"].map(normalize_text)
jobs_exp["job_skills_list"] = jobs_exp["Skills"].map(parse_skills)

jobs_exp["responsibilities_clean"] = jobs_exp["Responsibilities"].map(normalize_text)
jobs_exp["keywords_clean"] = jobs_exp["Keywords"].map(normalize_text)

# New text representation:
# Put the most important role-defining information first
jobs_exp["job_text_v2"] = (
    "Role title: " + jobs_exp["job_title"] + ". "
    "Experience level: " + jobs_exp["experience_level"] + ". "
    "Required years of experience: " + jobs_exp["years_of_experience"] + ". "
    "Core required skills: " + jobs_exp["job_skills_list"].map(lambda xs: ", ".join(xs)) + ". "
    "Responsibilities: " + jobs_exp["responsibilities_clean"] + ". "
    "Keywords: " + jobs_exp["keywords_clean"] + ". "
    "This role is related to " + jobs_exp["normalized_job_title"] + "."
)

jobs_exp = jobs_exp[
    [
        "job_id",
        "job_title",
        "normalized_job_title",
        "experience_level",
        "years_of_experience",
        "job_skills_list",
        "job_text_v2"
    ]
].copy()

print("jobs_exp shape:", jobs_exp.shape)
display(jobs_exp.head(3))

jobs_exp shape: (1068, 7)


,job_id,job_title,normalized_job_title,experience_level,years_of_experience,job_skills_list,job_text_v2
0,NET-F-001,.NET Developer,.net developer,fresher,0-1,"[c#, vb.net, .net framework, .net core, asp.net, mvc, html, css, javascript, sql server, entity framework, linq, visual studio, git, unit testing]","Role title: .NET Developer. Experience level: fresher. Required years of experience: 0-1. Core required skills: c#, vb.net, .net framework, .net core, asp.net, mvc, html, css, javascript, sql server, entity framework, linq, visual studio, git, unit testing. Responsibilities: Assist in coding and debugging applications; Learn and apply .NET Framework and Core fundamentals; Support team in building ASP.NET MVC web applications; Write basic SQL queries and work with Entity Framework; Collaborate with peers to solve issues; Participate in code reviews for learning; Follow best practices in coding; Work with version control (Git). Keywords: .NET; C#; ASP.NET MVC; Entity Framework; SQL Server; LINQ; Visual Studio; Unit Testing. This role is related to .net developer."
1,NET-F-002,.NET Developer,.net developer,fresher,0-1,"[c#, .net framework, asp.net, razor, html, css, javascript, sql server, entity framework, nunit]","Role title: .NET Developer. Experience level: fresher. Required years of experience: 0-1. Core required skills: c#, .net framework, asp.net, razor, html, css, javascript, sql server, entity framework, nunit. Responsibilities: Write simple C# programs under guidance; Support development of ASP.NET MVC applications; Implement Razor views and front-end logic; Assist in database query writing; Participate in unit testing tasks; Learn and apply LINQ for data operations; Work with mentors for code corrections. Keywords: .NET; C#; ASP.NET MVC; Entity Framework; SQL Server; Razor; Unit Testing. This role is related to .net developer."
2,NET-F-003,.NET Developer,.net developer,fresher,0-1,"[c#, vb.net, .net core, asp.net mvc, html, css, javascript, sql server, git]","Role title: .NET Developer. Experience level: fresher. Required years of experience: 0-1. Core required skills: c#, vb.net, .net core, asp.net mvc, html, css, javascript, sql server, git. Responsibilities: Contribute to development of small modules; Assist in bug fixing and debugging; Learn and implement MVC patterns; Support database integration tasks; Understand version control basics; Work on minor testing scripts; Follow coding standards. Keywords: .NET; C#; ASP.NET MVC; SQL Server; Entity Framework; Git. This role is related to .net developer."


In [5]:
# -----------------------------
# Build a cleaner resumes table
# -----------------------------

resumes_exp = resumes.copy().reset_index(drop=True)

# Create stable IDs
resumes_exp["resume_id"] = ["R_%04d" % i for i in range(len(resumes_exp))]

# Fill expected missing fields
for col in ["Current_Job_Title", "Previous_Job_Titles", "Certifications", "Target_Job_Description"]:
    resumes_exp[col] = resumes_exp[col].fillna("")

# Canonical fields
resumes_exp["current_job_title"] = resumes_exp["Current_Job_Title"].map(normalize_text)
resumes_exp["normalized_resume_title"] = resumes_exp["current_job_title"].map(normalize_title)

resumes_exp["experience_years"] = resumes_exp["Experience_Years"].astype(int)
resumes_exp["resume_skills_list"] = resumes_exp["Skills"].map(parse_skills)

resumes_exp["education_level_clean"] = resumes_exp["Education_Level"].map(normalize_text)
resumes_exp["field_of_study_clean"] = resumes_exp["Field_of_Study"].map(normalize_text)
resumes_exp["degrees_clean"] = resumes_exp["Degrees"].map(normalize_text)
resumes_exp["institute_clean"] = resumes_exp["Institute_Name"].map(normalize_text)
resumes_exp["certifications_clean"] = resumes_exp["Certifications"].map(normalize_text)
resumes_exp["target_job_description"] = resumes_exp["Target_Job_Description"].map(normalize_text)

# Fresher / experienced flag
resumes_exp["career_stage"] = np.where(
    resumes_exp["experience_years"] == 0,
    "fresher",
    "experienced"
)

# New text representation:
# For freshers, education + target role + skills are especially important
resumes_exp["resume_text_v2"] = (
    "Candidate career stage: " + resumes_exp["career_stage"] + ". "
    "Years of experience: " + resumes_exp["experience_years"].astype(str) + ". "
    "Current role: " + resumes_exp["current_job_title"] + ". "
    "Education level: " + resumes_exp["education_level_clean"] + ". "
    "Degree: " + resumes_exp["degrees_clean"] + ". "
    "Field of study: " + resumes_exp["field_of_study_clean"] + ". "
    "Institution: " + resumes_exp["institute_clean"] + ". "
    "Skills: " + resumes_exp["resume_skills_list"].map(lambda xs: ", ".join(xs)) + ". "
    "Certifications: " + resumes_exp["certifications_clean"] + ". "
    "Target job description: " + resumes_exp["target_job_description"] + ". "
    "Preferred role direction based on candidate background."
)

resumes_exp = resumes_exp[
    [
        "resume_id",
        "current_job_title",
        "normalized_resume_title",
        "experience_years",
        "career_stage",
        "resume_skills_list",
        "target_job_description",
        "resume_text_v2"
    ]
].copy()

print("resumes_exp shape:", resumes_exp.shape)
display(resumes_exp.head(3))

resumes_exp shape: (1200, 8)


,resume_id,current_job_title,normalized_resume_title,experience_years,career_stage,resume_skills_list,target_job_description,resume_text_v2
0,R_0000,,,0,fresher,"[node.js, javascript, deep learning, statistics, sql]",Seeking a challenging role as a Software Developer where I can apply my skills and knowledge to contribute to organizational success and professional growth.,"Candidate career stage: fresher. Years of experience: 0. Current role: . Education level: Master's. Degree: Master's in Cybersecurity. Field of study: Cybersecurity. Institution: University of Pennsylvania. Skills: node.js, javascript, deep learning, statistics, sql. Certifications: Google Cloud Professional. Target job description: Seeking a challenging role as a Software Developer where I can apply my skills and knowledge to contribute to organizational success and professional growth.. Preferred role direction based on candidate background."
1,R_0001,Cybersecurity Engineer,cybersecurity engineer,5,experienced,"[spark, kubernetes, terraform, natural language processing]",Targeting a Cybersecurity Engineer position to utilize my educational background and experience to drive results and achieve career objectives.,"Candidate career stage: experienced. Years of experience: 5. Current role: Cybersecurity Engineer. Education level: Bachelor's. Degree: Bachelor's in Electronics Engineering. Field of study: Electronics Engineering. Institution: Pune University. Skills: spark, kubernetes, terraform, natural language processing. Certifications: TensorFlow Developer Certificate. Target job description: Targeting a Cybersecurity Engineer position to utilize my educational background and experience to drive results and achieve career objectives.. Preferred role direction based on candidate background."
2,R_0002,Prompt Engineer,prompt engineer,2,experienced,"[data analysis, node.js, machine learning, linux, jenkins, network security, rest apis]",Targeting a Prompt Engineer position to utilize my educational background and experience to drive results and achieve career objectives.,"Candidate career stage: experienced. Years of experience: 2. Current role: Prompt Engineer. Education level: Bachelor's. Degree: Bachelor's in Computer Science. Field of study: Computer Science. Institution: Amity University. Skills: data analysis, node.js, machine learning, linux, jenkins, network security, rest apis. Certifications: Microsoft Azure Fundamentals, Cisco Certified Network Associate, Oracle Certified Professional. Target job description: Targeting a Prompt Engineer position to utilize my educational background and experience to drive results and achieve career objectives.. Preferred role direction based on candidate background."


In [6]:
# -----------------------------
# Sanity checks
# -----------------------------

print("Empty job_text_v2:", (jobs_exp["job_text_v2"].str.len() == 0).sum())
print("Empty resume_text_v2:", (resumes_exp["resume_text_v2"].str.len() == 0).sum())

print("\njob_text_v2 length stats:")
print(jobs_exp["job_text_v2"].str.len().describe())

print("\nresume_text_v2 length stats:")
print(resumes_exp["resume_text_v2"].str.len().describe())

Empty job_text_v2: 0
Empty resume_text_v2: 0

job_text_v2 length stats:
count    1068.000000
mean      707.138577
std       187.607527
min       361.000000
25%       546.750000
50%       688.000000
75%       864.250000
max      1395.000000
Name: job_text_v2, dtype: float64

resume_text_v2 length stats:
count    1200.000000
mean      612.753333
std        50.887511
min       475.000000
25%       575.000000
50%       613.000000
75%       646.000000
max       814.000000
Name: resume_text_v2, dtype: float64


In [7]:
# -----------------------------
# Save the new experiment tables
# -----------------------------

jobs_exp.to_parquet(DATA_PROCESSED / "jobs_exp_v2.parquet", index=False)
resumes_exp.to_parquet(DATA_PROCESSED / "resumes_exp_v2.parquet", index=False)

print("Saved experiment parquet files.")

Saved experiment parquet files.


In [8]:
# -----------------------------
# Generate embeddings for the new text representation
# -----------------------------

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

job_texts_v2 = jobs_exp["job_text_v2"].astype(str).tolist()
resume_texts_v2 = resumes_exp["resume_text_v2"].astype(str).tolist()

job_emb_v2 = model.encode(
    job_texts_v2,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

resume_emb_v2 = model.encode(
    resume_texts_v2,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("job_emb_v2 shape:", job_emb_v2.shape)
print("resume_emb_v2 shape:", resume_emb_v2.shape)

C:\Users\COMPUTER CARE\anaconda3\envs\jobplatform\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

job_emb_v2 shape: (1068, 384)
resume_emb_v2 shape: (1200, 384)


In [9]:
# -----------------------------
# Save embeddings locally
# -----------------------------

np.save(DATA_PROCESSED / "job_emb_v2.npy", job_emb_v2)
np.save(DATA_PROCESSED / "resume_emb_v2.npy", resume_emb_v2)

print("Saved v2 embeddings.")

Saved v2 embeddings.


In [10]:
# -----------------------------
# Experienced evaluation set
# -----------------------------

# Ground truth for experienced users = current normalized title
resumes_exp["ground_truth_title"] = resumes_exp["normalized_resume_title"]

experienced_eval = resumes_exp[resumes_exp["ground_truth_title"] != ""].copy()

print("Experienced eval size:", len(experienced_eval))

Experienced eval size: 759


In [11]:
# -----------------------------
# Infer target role for freshers
# -----------------------------

def infer_title_from_target_desc(text: str):
    """
    Extract a target role from free-text target description.
    This is heuristic, but useful for proxy evaluation.
    """
    if not text:
        return ""

    t = text.lower()

    patterns = [
        r"role as a ([a-zA-Z ]+)",
        r"role as an ([a-zA-Z ]+)",
        r"targeting a ([a-zA-Z ]+) position",
        r"targeting an ([a-zA-Z ]+) position",
        r"seeking a ([a-zA-Z ]+) role",
        r"seeking an ([a-zA-Z ]+) role",
        r"looking for a ([a-zA-Z ]+) role",
        r"looking for an ([a-zA-Z ]+) role",
        r"position as a ([a-zA-Z ]+)",
        r"position as an ([a-zA-Z ]+)",
    ]

    for pat in patterns:
        m = re.search(pat, t)
        if m:
            candidate = m.group(1).strip()
            return normalize_title(candidate)

    return ""

freshers_df = resumes_exp[
    (resumes_exp["experience_years"] == 0) | (resumes_exp["normalized_resume_title"] == "")
].copy()

freshers_df["ground_truth_title"] = freshers_df["target_job_description"].apply(infer_title_from_target_desc)
freshers_eval = freshers_df[freshers_df["ground_truth_title"] != ""].copy()

print("Freshers eval size:", len(freshers_eval))
display(freshers_eval[["resume_id", "target_job_description", "ground_truth_title"]].head(10))

Freshers eval size: 266


,resume_id,target_job_description,ground_truth_title
0,R_0000,Seeking a challenging role as a Software Developer where I can apply my skills and knowledge to contribute to organizational success and professional growth.,software developer where i can apply my skills and knowledge to contribute to organizational success and professional growth
5,R_0005,Targeting a Quantum Computing Specialist position to utilize my educational background and experience to drive results and achieve career objectives.,quantum computing specialist
6,R_0006,Seeking a challenging role as a Backend Developer where I can apply my skills and knowledge to contribute to organizational success and professional growth.,backend developer where i can apply my skills and knowledge to contribute to organizational success and professional growth
15,R_0015,Seeking a challenging role as a Quantum Computing Specialist where I can apply my skills and knowledge to contribute to organizational success and professional growth.,quantum computing specialist where i can apply my skills and knowledge to contribute to organizational success and professional growth
20,R_0020,Seeking a challenging role as a Mobile Applications Developer where I can apply my skills and knowledge to contribute to organizational success and professional growth.,mobile applications developer where i can apply my skills and knowledge to contribute to organizational success and professional growth
28,R_0028,"Looking for a Cloud Engineer role where I can architect cloud solutions, automate deployments, and optimize cloud resources for performance and cost-efficiency.",cloud engineer
33,R_0033,Seeking a challenging role as a AI Ethics Officer where I can apply my skills and knowledge to contribute to organizational success and professional growth.,ai ethics officer where i can apply my skills and knowledge to contribute to organizational success and professional growth
38,R_0038,Seeking a challenging role as a Blockchain Engineer where I can apply my skills and knowledge to contribute to organizational success and professional growth.,blockchain engineer where i can apply my skills and knowledge to contribute to organizational success and professional growth
40,R_0040,Targeting a Cybersecurity Engineer position to utilize my educational background and experience to drive results and achieve career objectives.,cybersecurity engineer
47,R_0047,"Targeting a Data Scientist position where I can utilize my expertise in Python, R, SQL, and machine learning algorithms to solve business challenges through data analysis and predictive modeling.",data scientist


In [12]:
# -----------------------------
# Ranking logic for v2 model
# -----------------------------

def experience_penalty(resume_years: int, job_min):
    """
    Soft penalty:
    - no penalty if the resume meets the minimum
    - reduced score if below minimum
    """
    if job_min is None:
        return 1.0
    if resume_years < job_min:
        gap = job_min - resume_years
        return max(0.4, 1.0 - 0.12 * gap)
    return 1.0

# Parse numeric min experience for jobs
jobs_exp["min_years"], jobs_exp["max_years"] = zip(*jobs_exp["years_of_experience"].map(parse_years_range))

def rank_jobs_baseline_v2(resume_idx: int):
    """
    Pure semantic similarity ranking.
    """
    sims = cosine_similarity(
        resume_emb_v2[resume_idx:resume_idx+1],
        job_emb_v2
    )[0]
    return np.argsort(sims)[::-1]

def rank_jobs_enhanced_v2(resume_idx: int, w_sem=0.8, w_exp=0.2):
    """
    Tuned enhanced ranking:
    final = 0.8 * semantic + 0.2 * (semantic * experience_penalty)
    """
    sims = cosine_similarity(
        resume_emb_v2[resume_idx:resume_idx+1],
        job_emb_v2
    )[0]

    resume_years = int(resumes_exp.loc[resume_idx, "experience_years"])

    penalties = np.array([
        experience_penalty(resume_years, mn)
        for mn in jobs_exp["min_years"]
    ])

    final = (w_sem * sims) + (w_exp * (sims * penalties))
    return np.argsort(final)[::-1]

In [13]:
# -----------------------------
# Evaluation functions
# -----------------------------

def precision_at_k_from_ranker_on_df(ranker_fn, eval_df, k=5):
    hits = 0
    total = 0

    for idx in eval_df.index:
        ranked_idx = ranker_fn(idx)
        top_k = ranked_idx[:k]

        gt = eval_df.loc[idx, "ground_truth_title"]
        predicted_titles = jobs_exp.iloc[top_k]["normalized_job_title"].values

        if gt in predicted_titles:
            hits += 1

        total += 1

    return hits / total if total > 0 else 0


def mrr_from_ranker_on_df(ranker_fn, eval_df):
    rrs = []

    for idx in eval_df.index:
        ranked_idx = ranker_fn(idx)
        gt = eval_df.loc[idx, "ground_truth_title"]

        ranked_titles = jobs_exp.iloc[ranked_idx]["normalized_job_title"].values

        rr = 0.0
        for i, title in enumerate(ranked_titles):
            if title == gt:
                rr = 1.0 / (i + 1)
                break

        rrs.append(rr)

    return float(np.mean(rrs))


def top1_acc_from_ranker_on_df(ranker_fn, eval_df):
    correct = 0
    total = 0

    for idx in eval_df.index:
        ranked_idx = ranker_fn(idx)
        top1 = ranked_idx[0]

        pred = jobs_exp.loc[top1, "normalized_job_title"]
        gt = eval_df.loc[idx, "ground_truth_title"]

        if pred == gt:
            correct += 1

        total += 1

    return correct / total if total > 0 else 0


def eval_suite_on_df(ranker_fn, label, eval_df, group_name):
    """
    Run a full evaluation suite on any subset.
    """
    return {
        "group": group_name,
        "model": label,
        "n": len(eval_df),
        "P@1": precision_at_k_from_ranker_on_df(ranker_fn, eval_df, k=1),
        "P@3": precision_at_k_from_ranker_on_df(ranker_fn, eval_df, k=3),
        "P@5": precision_at_k_from_ranker_on_df(ranker_fn, eval_df, k=5),
        "P@10": precision_at_k_from_ranker_on_df(ranker_fn, eval_df, k=10),
        "MRR": mrr_from_ranker_on_df(ranker_fn, eval_df),
        "Top1_Acc": top1_acc_from_ranker_on_df(ranker_fn, eval_df),
    }

In [14]:
# -----------------------------
# Evaluate experienced users
# -----------------------------

exp_baseline_v2 = eval_suite_on_df(
    rank_jobs_baseline_v2,
    "Baseline v2",
    experienced_eval,
    "Experienced"
)

exp_enhanced_v2 = eval_suite_on_df(
    lambda idx: rank_jobs_enhanced_v2(idx, 0.8, 0.2),
    "Enhanced v2 (0.8,0.2)",
    experienced_eval,
    "Experienced"
)

exp_results_v2 = pd.DataFrame([exp_baseline_v2, exp_enhanced_v2])
display(exp_results_v2)

,group,model,n,P@1,P@3,P@5,P@10,MRR,Top1_Acc
0,Experienced,Baseline v2,759,0.255599,0.300395,0.314888,0.325428,0.282586,0.255599
1,Experienced,"Enhanced v2 (0.8,0.2)",759,0.262187,0.303030,0.313570,0.326746,0.288262,0.262187


In [15]:
# -----------------------------
# Evaluate freshers
# -----------------------------

fresh_baseline_v2 = eval_suite_on_df(
    rank_jobs_baseline_v2,
    "Baseline v2",
    freshers_eval,
    "Freshers"
)

fresh_enhanced_v2 = eval_suite_on_df(
    lambda idx: rank_jobs_enhanced_v2(idx, 0.8, 0.2),
    "Enhanced v2 (0.8,0.2)",
    freshers_eval,
    "Freshers"
)

fresh_results_v2 = pd.DataFrame([fresh_baseline_v2, fresh_enhanced_v2])
display(fresh_results_v2)

,group,model,n,P@1,P@3,P@5,P@10,MRR,Top1_Acc
0,Freshers,Baseline v2,266,0.135338,0.165414,0.176692,0.195489,0.157116,0.135338
1,Freshers,"Enhanced v2 (0.8,0.2)",266,0.146617,0.161654,0.187970,0.203008,0.165219,0.146617


In [16]:
# -----------------------------
# Compare against previous best model
# -----------------------------

previous_results = pd.DataFrame([
    {
        "group": "Experienced",
        "model": "Previous Enhanced Best",
        "n": 759,
        "P@1": 0.3215,
        "P@3": None,
        "P@5": 0.3781,
        "P@10": None,
        "MRR": 0.3483,
        "Top1_Acc": 0.3215
    },
    {
        "group": "Freshers",
        "model": "Previous Enhanced Best",
        "n": 266,
        "P@1": 0.1090,
        "P@3": None,
        "P@5": 0.1880,
        "P@10": 0.2143,
        "MRR": 0.1449,
        "Top1_Acc": 0.1090
    }
])

current_results = pd.concat([exp_results_v2, fresh_results_v2], ignore_index=True)

display(previous_results)
display(current_results)

,group,model,n,P@1,P@3,P@5,P@10,MRR,Top1_Acc
0,Experienced,Previous Enhanced Best,759,0.3215,None,0.3781,NaN,0.3483,0.3215
1,Freshers,Previous Enhanced Best,266,0.1090,None,0.1880,0.2143,0.1449,0.1090


,group,model,n,P@1,P@3,P@5,P@10,MRR,Top1_Acc
0,Experienced,Baseline v2,759,0.255599,0.300395,0.314888,0.325428,0.282586,0.255599
1,Experienced,"Enhanced v2 (0.8,0.2)",759,0.262187,0.303030,0.313570,0.326746,0.288262,0.262187
2,Freshers,Baseline v2,266,0.135338,0.165414,0.176692,0.195489,0.157116,0.135338
3,Freshers,"Enhanced v2 (0.8,0.2)",266,0.146617,0.161654,0.187970,0.203008,0.165219,0.146617


In [17]:
# -----------------------------
# Manual inspection of one resume
# -----------------------------

def show_top_jobs_for_resume_v2(resume_id, k=10):
    """
    Show top-k jobs for a chosen resume under the v2 representation.
    Useful for manual checking of recommendation quality.
    """
    r_idx = resumes_exp.index[resumes_exp["resume_id"] == resume_id][0]
    ranked_idx = rank_jobs_enhanced_v2(r_idx, 0.8, 0.2)[:k]

    out = jobs_exp.iloc[ranked_idx][["job_id", "job_title", "experience_level", "years_of_experience"]].copy()

    sims = cosine_similarity(
        resume_emb_v2[r_idx:r_idx+1],
        job_emb_v2
    )[0]

    penalties = np.array([
        experience_penalty(int(resumes_exp.loc[r_idx, "experience_years"]), mn)
        for mn in jobs_exp["min_years"]
    ])

    final = (0.8 * sims) + (0.2 * (sims * penalties))

    out["semantic_score"] = sims[ranked_idx]
    out["final_score"] = final[ranked_idx]

    print("Resume ID:", resume_id)
    print("Career stage:", resumes_exp.loc[r_idx, "career_stage"])
    print("Experience years:", resumes_exp.loc[r_idx, "experience_years"])
    print("Skills:", resumes_exp.loc[r_idx, "resume_skills_list"])
    print("Target description:", resumes_exp.loc[r_idx, "target_job_description"])
    display(out.reset_index(drop=True))

show_top_jobs_for_resume_v2("R_0000", k=10)

Resume ID: R_0000
Career stage: fresher
Experience years: 0
Skills: ['node.js', 'javascript', 'deep learning', 'statistics', 'sql']
Target description: Seeking a challenging role as a Software Developer where I can apply my skills and knowledge to contribute to organizational success and professional growth.


,job_id,job_title,experience_level,years_of_experience,semantic_score,final_score
0,FE-F-002,Fintech Engineer,fresher,0-1,0.678913,0.678913
1,FE-F-001,Fintech Engineer,fresher,0-1,0.677994,0.677994
2,FE-F-005,Fintech Engineer,fresher,0-1,0.676973,0.676973
3,FE-F-006,Fintech Engineer,fresher,0-1,0.675371,0.675371
4,CSA-F-001,Cybersecurity Analyst,fresher,0-1,0.672741,0.672741
5,FE-F-008,Fintech Engineer,fresher,0-1,0.670455,0.670455
6,FE-F-009,Fintech Engineer,fresher,0-1,0.670350,0.670350
7,FE-F-007,Fintech Engineer,fresher,0-1,0.670266,0.670266
8,FE-F-004,Fintech Engineer,fresher,0-1,0.669992,0.669992
9,FE-F-003,Fintech Engineer,fresher,0-1,0.668646,0.668646
